### Bronze → Silver: decode GTFS-Realtime protobuf into typed stop-level rows

- **Decode:** `from_protobuf` using a compiled descriptor (`.desc`) — decodes in the engine (JVM),
  so **executors need no Python `gtfs-realtime-bindings`** (avoids the `ModuleNotFoundError` from UDFs).
- **Grain:** one row per `stop_time_update` (explode `entity[]` → `stop_time_update[]`).
- **Incremental:** streaming read of bronze + checkpoint → each run decodes only newly-appended rows.

**Run this as a serverless job/notebook — not via Databricks Connect** (Structured Streaming isn't
supported over Spark Connect). The one-time descriptor build needs `gtfs-realtime-bindings` in the
job environment; after the `.desc` exists it is never imported again.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.protobuf.functions import from_protobuf

# Catalog is passed as a job parameter (default -> ${var.catalog} per target); the default here
# keeps interactive runs working. Schemas/volumes are the same across dev and prod.
dbutils.widgets.text("catalog", "transport_vic_dev")
CATALOG = dbutils.widgets.get("catalog")

BRONZE_TBL = f"{CATALOG}.`01_bronze`.vline_trip_updates"
SILVER_TBL = f"{CATALOG}.`02_silver`.trip_updates"
CHECKPOINT = f"/Volumes/{CATALOG}/02_silver/_checkpoints/trip_updates"
DESC_PATH  = f"/Volumes/{CATALOG}/02_silver/artifacts/gtfs_realtime.desc"
MESSAGE    = "transit_realtime.FeedMessage"

In [ ]:
# One-time setup: schema + volumes for the checkpoint and the protobuf descriptor.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.`02_silver`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.`02_silver`._checkpoints")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.`02_silver`.artifacts")

In [ ]:
# Build the FileDescriptorSet ONCE from the installed bindings and cache it in the Volume.
# Runs on the driver (needs gtfs-realtime-bindings); skipped entirely once the .desc exists.
import os

if not os.path.exists(DESC_PATH):
    from google.protobuf import descriptor_pb2
    from google.transit import gtfs_realtime_pb2

    fds, seen = descriptor_pb2.FileDescriptorSet(), set()
    def _add(fd):
        if fd.name in seen:
            return
        seen.add(fd.name)
        for dep in fd.dependencies:
            _add(dep)
        fd.CopyToProto(fds.file.add())

    _add(gtfs_realtime_pb2.DESCRIPTOR)
    with open(DESC_PATH, "wb") as f:
        f.write(fds.SerializeToString())
    print("wrote descriptor ->", DESC_PATH)
else:
    print("descriptor already present ->", DESC_PATH)

In [ ]:
# Pre-create silver with explicit types, NOT NULL on the always-present keys, and liquid clustering.
# Everything under arrival/departure stays nullable (independently optional in the feed).
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TBL} (
  feed_ts             TIMESTAMP NOT NULL,
  entity_id           STRING    NOT NULL,
  trip_id             STRING    NOT NULL,
  route_id            STRING,
  start_date          DATE,
  start_time          STRING,               -- keep as string: GTFS allows >24:00:00
  trip_rel            STRING,
  stop_sequence       INT,                  -- sparse / non-contiguous within a trip
  stop_id             STRING,
  arrival_time        TIMESTAMP,
  arrival_delay_sec   INT,
  departure_time      TIMESTAMP,
  departure_delay_sec INT,
  stop_rel            STRING,
  event_time          TIMESTAMP,            -- coalesce(arrival, departure) for sorting/clustering
  source_file         STRING    NOT NULL,
  _ingest_ts          TIMESTAMP,
  _silver_ts          TIMESTAMP
)
USING DELTA
CLUSTER BY (start_date, route_id)
""")

In [ ]:
# Streaming decode. from_protobuf gives a properly-typed struct: time fields are LONG epoch
# seconds (cast straight to timestamp), delays are INT, enums are their string names.
bronze = (spark.readStream
          .option("skipChangeCommits", "true")   # tolerate future OPTIMIZE on bronze
          .table(BRONZE_TBL))

decoded = bronze.select(
    F.col("path").alias("source_file"),
    F.col("_ingest_ts"),
    from_protobuf(F.col("content"), MESSAGE, descFilePath=DESC_PATH).alias("feed"),
)

entities = (decoded
    .withColumn("feed_ts", F.col("feed.header.timestamp").cast("timestamp"))
    .select("source_file", "_ingest_ts", "feed_ts", F.explode("feed.entity").alias("e")))

trips = entities.select(
    "source_file", "_ingest_ts", "feed_ts",
    F.col("e.id").alias("entity_id"),
    F.col("e.trip_update.trip.trip_id").alias("trip_id"),
    F.col("e.trip_update.trip.route_id").alias("route_id"),
    F.to_date("e.trip_update.trip.start_date", "yyyyMMdd").alias("start_date"),
    F.col("e.trip_update.trip.start_time").alias("start_time"),
    F.coalesce(F.col("e.trip_update.trip.schedule_relationship"), F.lit("SCHEDULED")).alias("trip_rel"),
    F.explode("e.trip_update.stop_time_update").alias("stu"),   # one generator per select
)

stops = (trips.select(
        "feed_ts", "entity_id", "trip_id", "route_id", "start_date", "start_time", "trip_rel",
        F.col("stu.stop_sequence").cast("int").alias("stop_sequence"),
        F.col("stu.stop_id").alias("stop_id"),
        F.col("stu.arrival.time").cast("timestamp").alias("arrival_time"),
        F.col("stu.arrival.delay").cast("int").alias("arrival_delay_sec"),
        F.col("stu.departure.time").cast("timestamp").alias("departure_time"),
        F.col("stu.departure.delay").cast("int").alias("departure_delay_sec"),
        F.coalesce(F.col("stu.schedule_relationship"), F.lit("SCHEDULED")).alias("stop_rel"),
        "source_file", "_ingest_ts",
    )
    .withColumn("event_time", F.coalesce("arrival_time", "departure_time"))
    .withColumn("_silver_ts", F.current_timestamp())
    .select(  # match the table DDL column order exactly
        "feed_ts", "entity_id", "trip_id", "route_id", "start_date", "start_time", "trip_rel",
        "stop_sequence", "stop_id", "arrival_time", "arrival_delay_sec", "departure_time",
        "departure_delay_sec", "stop_rel", "event_time", "source_file", "_ingest_ts", "_silver_ts",
    ))

In [ ]:
# Append new rows, then stop. Separate checkpoint from bronze (one per query).
q = (stops.writeStream
     .option("checkpointLocation", CHECKPOINT)
     .trigger(availableNow=True)
     .toTable(SILVER_TBL))
q.awaitTermination()
print("silver rows:", spark.table(SILVER_TBL).count())

#### Notes
- **Natural grain / key:** `(feed_ts, entity_id, stop_sequence)`. Because the feed is `FULL_DATASET`,
  every snapshot re-publishes each trip — silver keeps that time-series (delay evolving over time).
  Do any "latest predicted arrival per stop" collapse in **gold**, not here.
- **Append is idempotent** thanks to the checkpoint (each bronze row decoded once) — no MERGE needed
  unless you reprocess from scratch.
- If `descFilePath` on a Volume ever misbehaves, read the bytes and pass `binaryDescriptorSet=` instead.